# Banco de Dados com SQLite

## Objetivo da aula

Nesta aula vamos aprender de forma bem simples o que é um banco de dados e como usar o SQLite em Python.

Para alunos de engenharia elétrica, a ideia é conectar o conceito de banco de dados com dados reais de medições, como:

- tensão
- corrente
- potência
- horário da medição
- local do equipamento

SQLite é uma ótima opção para começar porque:

- é leve
- não precisa de servidor
- salva tudo em um arquivo local
- funciona muito bem em projetos acadêmicos e protótipos

---

## 1. O que é um banco de dados?

Um banco de dados é um lugar organizado para guardar informações. Em vez de salvar tudo em vários arquivos CSV ou em planilhas, organizamos os dados em tabelas.

Exemplo:

- tabela `medicao`: guarda cada leitura elétrica
- tabela `equipamento`: guarda os nomes e locais dos equipamentos

Isso facilita procurar, filtrar e analisar dados.

---

## 2. Quando usar banco de dados?

Você usa banco de dados quando precisa:

- guardar muitas informações
- consultar dados rapidamente
- manter a organização dos dados
- evitar erros de manipulação manual

Em engenharia elétrica, isso pode ser usado para:

- registrar medições de consumo
- acompanhar dados de sensores
- armazenar histórico de temperatura, corrente e tensão
- comparar desempenho de equipamentos

---

## 3. SQLite: o banco de dados simples e poderoso

O SQLite é um banco de dados relacional que salva tudo em um arquivo, como por exemplo:

`dados_energia.db`

Ele é muito útil em projetos didáticos porque:

- não exige instalação de servidor
- é fácil de aprender
- funciona com Python usando o módulo `sqlite3`

---

## 4. Conceitos básicos

Antes de começar, vamos entender alguns termos:

- `banco de dados`: conjunto de dados organizados
- `tabela`: estrutura para guardar dados de um tipo
- `linha`: um registro
- `coluna`: uma informação da tabela
- `consulta`: pergunta feita ao banco

Exemplo de tabela:

| id | horario | tensao_V | corrente_A |
|----|---------|----------|------------|
| 1  | 08:00   | 220      | 2.0        |
| 2  | 09:00   | 218      | 2.4        |

---

## 5. Primeiro passo: criar o banco

Vamos criar um banco de dados em Python e uma tabela para registrar medições elétricas.

In [1]:
import sqlite3

# Conectando ao banco de dados
# Se o arquivo não existir, o SQLite cria automaticamente
conn = sqlite3.connect('dados_energia.db')
cursor = conn.cursor()

print('Conexão com SQLite realizada com sucesso!')
print('Arquivo do banco: dados_energia.db')

Conexão com SQLite realizada com sucesso!
Arquivo do banco: dados_energia.db


## 6. Criando uma tabela

A tabela `medicao` vai guardar informações de cada leitura feita em um sistema elétrico.

Vamos definir algumas colunas:

- `id`: identificador único
- `horario`: hora da medição
- `tensao_V`: tensão em volts
- `corrente_A`: corrente em amperes
- `potencia_kW`: potência em kW

Vamos criar a tabela.

In [2]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS medicao (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    horario TEXT,
    tensao_V REAL,
    corrente_A REAL,
    potencia_kW REAL
)
''')

conn.commit()
print('Tabela medicao criada com sucesso!')

Tabela medicao criada com sucesso!


## 7. Inserindo dados de medição

Agora vamos inserir algumas leituras de um sistema elétrico. É comum ter dados coletados em horários diferentes.

A estrutura de uma inserção em SQL é:

`INSERT INTO nome_tabela (colunas) VALUES (valores)`

Vamos inserir alguns valores de exemplo.

In [3]:
medicoes = [
    ('08:00', 220.0, 2.0, 0.44),
    ('09:00', 218.0, 2.4, 0.52),
    ('10:00', 221.0, 1.8, 0.40),
    ('11:00', 219.0, 2.2, 0.48),
    ('12:00', 223.0, 2.1, 0.47),
    ('13:00', 217.0, 2.6, 0.56),
]

cursor.executemany('''
INSERT INTO medicao (horario, tensao_V, corrente_A, potencia_kW)
VALUES (?, ?, ?, ?)
''', medicoes)

conn.commit()
print('Dados inseridos com sucesso!')

Dados inseridos com sucesso!


## 8. Consultando os dados

Para ver os dados armazenados, usamos o comando `SELECT`.

Vamos consultar todas as linhas da tabela.

In [4]:
for linha in cursor.execute('SELECT id, horario, tensao_V, corrente_A, potencia_kW FROM medicao ORDER BY horario'):
    print(linha)

(1, '08:00', 220.0, 2.0, 0.44)
(2, '09:00', 218.0, 2.4, 0.52)
(3, '10:00', 221.0, 1.8, 0.4)
(4, '11:00', 219.0, 2.2, 0.48)
(5, '12:00', 223.0, 2.1, 0.47)
(6, '13:00', 217.0, 2.6, 0.56)


## 9. Filtrando dados

Nem sempre precisamos ver tudo. Podemos buscar apenas registros que atendam a algum critério.

Por exemplo:

- tensão maior que 220 V
- corrente maior que 2.0 A
- valores em determinado horário

Vamos fazer um filtro simples.

In [5]:
consulta = '''
SELECT horario, tensao_V, corrente_A
FROM medicao
WHERE tensao_V > 220
ORDER BY horario
'''

for linha in cursor.execute(consulta):
    print(linha)

('10:00', 221.0, 1.8)
('12:00', 223.0, 2.1)


## 10. Usando funções de agregação

Banco de dados também nos ajuda a calcular valores importantes.

Podemos usar funções como:

- `AVG()`: média
- `MAX()`: maior valor
- `MIN()`: menor valor
- `SUM()`: soma

Vamos calcular a média da tensão e a maior corrente.

In [6]:
consulta_agg = '''
SELECT AVG(tensao_V) AS media_tensao,
       MAX(corrente_A) AS maior_corrente,
       AVG(potencia_kW) AS media_potencia
FROM medicao
'''

resultado = cursor.execute(consulta_agg).fetchone()
print('Média da tensão:', resultado[0])
print('Maior corrente:', resultado[1])
print('Média da potência:', resultado[2])

Média da tensão: 219.66666666666666
Maior corrente: 2.6
Média da potência: 0.47833333333333333


## 11. Atualizando dados

Às vezes, uma medição pode ter sido registrada com valor errado. Então usamos o comando `UPDATE`.

Vamos corrigir a corrente do horário `10:00`.

In [7]:
cursor.execute('''
UPDATE medicao
SET corrente_A = 2.0
WHERE horario = '10:00'
''')

conn.commit()
print('Dados atualizados com sucesso!')

for linha in cursor.execute('SELECT horario, tensao_V, corrente_A FROM medicao WHERE horario = "10:00"'):
    print(linha)

Dados atualizados com sucesso!
('10:00', 221.0, 2.0)


## 12. Removendo dados

Também podemos apagar um registro específico com `DELETE`.

Vamos remover uma medição de exemplo.

In [8]:
cursor.execute('''
DELETE FROM medicao
WHERE horario = '13:00'
''')

conn.commit()
print('Registro removido com sucesso!')

for linha in cursor.execute('SELECT * FROM medicao'):
    print(linha)

Registro removido com sucesso!
(1, '08:00', 220.0, 2.0, 0.44)
(2, '09:00', 218.0, 2.4, 0.52)
(3, '10:00', 221.0, 2.0, 0.4)
(4, '11:00', 219.0, 2.2, 0.48)
(5, '12:00', 223.0, 2.1, 0.47)


## 13. Relacionando tabelas

Até agora usamos uma única tabela. Mas em bancos de dados reais, normalmente temos mais de uma tabela.

Vamos criar uma segunda tabela chamada `equipamento`, que guardará:

- `id`
- `nome`
- `local`

Depois, podemos relacionar cada medição a um equipamento.

In [9]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS equipamento (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT,
    local TEXT
)
''')

cursor.execute('''
INSERT INTO equipamento (nome, local)
VALUES ('Motor 01', 'Laboratório de Energia'),
       ('Carga Resistiva', 'Sala de Testes')
''')

conn.commit()
print('Tabela equipamento criada e populada!')

for linha in cursor.execute('SELECT * FROM equipamento'):
    print(linha)

Tabela equipamento criada e populada!
(1, 'Motor 01', 'Laboratório de Energia')
(2, 'Carga Resistiva', 'Sala de Testes')


## 14. Integrando SQLite com pandas

O pandas é muito usado em análise de dados. Podemos ler uma tabela do SQLite diretamente para um DataFrame.

Isso é útil quando queremos visualizar e analisar dados em Python com gráficos ou estatísticas.

In [10]:
import pandas as pd

df = pd.read_sql_query('SELECT * FROM medicao', conn)
print(df)

   id horario  tensao_V  corrente_A  potencia_kW
0   1   08:00     220.0         2.0         0.44
1   2   09:00     218.0         2.4         0.52
2   3   10:00     221.0         2.0         0.40
3   4   11:00     219.0         2.2         0.48
4   5   12:00     223.0         2.1         0.47


## 15. Vantagens do SQLite para engenharia

O SQLite é muito útil em projetos de engenharia porque permite:

- armazenar dados em arquivos locais
- registrar histórico de medições
- criar pequenos bancos para projetos acadêmicos
- integrar com Python, pandas e gráficos
- ser simples de manter e aprender

Em projetos maiores, pode ser usado como base inicial antes de migrar para PostgreSQL, MySQL ou outros bancos.

---

## 16. Resumo da aula

Na prática, aprendemos que:

- banco de dados organiza informações em tabelas
- SQLite é um banco leve e fácil de usar
- `sqlite3` permite conectar Python ao banco
- `CREATE TABLE`, `INSERT`, `SELECT`, `UPDATE` e `DELETE` são comandos fundamentais
- filtros e agregações ajudam a analisar dados
- pandas pode ler os dados do banco para análise

Este é o começo de um tema muito importante em programação e engenharia.

---

## 17. Exercício proposto

Crie uma tabela chamada `sensor` com as colunas:

- `id`
- `nome_sensor`
- `tipo`
- `valor`
- `horario`

Depois:

1. insira 5 registros de exemplo
2. mostre todos os dados
3. filtre pelos sensores do tipo `tensão`
4. calcule a média dos valores

Se conseguir fazer isso, você já está dominando os conceitos básicos de SQLite.

---

## 18. Dica final

Na próxima aula, você pode aprender:

- como usar `JOIN` para relacionar tabelas
- como criar bancos mais complexos
- como salvar dados de sensores em tempo real
- como conectar Python com arquivos CSV, Excel e banco de dados juntos

Pratique bastante, porque a melhor forma de aprender banco de dados é criar, consultar e corrigir dados.